# Clean cards from MTGJSON

Source: https://mtgjson.com/api/v5/, accessed 2026-01-03.

In [58]:
import pandas as pd

In [59]:
cards_df = pd.read_csv('data/mtgjson.com/cards.csv')

cards_df.head()

C:\Users\sophc\AppData\Local\Temp\ipykernel_48256\3689862584.py:1: DtypeWarning: Columns (0: asciiName, 1: attractionLights, 2: cardParts, 3: duelDeck, 4: faceFlavorName, 5: facePrintedName, 6: flavorName, 7: frameVersion, 8: hasAlternativeDeckLimit, 9: hasContentWarning, 10: isFunny, 11: isOnlineOnly, 12: isOversized, 13: isRebalanced, 14: isReserved, 15: isStorySpotlight, 16: isTextless, 17: isTimeshifted, 18: loyalty, 19: originalPrintings, 20: originalReleaseDate, 21: printedName, 22: printedText, 23: printedType, 24: rebalancedPrintings, 25: relatedCards, 26: signature, 27: subsets) have mixed types. Specify dtype option on import or set low_memory=False.
  cards_df = pd.read_csv('data/mtgjson.com/cards.csv')


,artist,artistIds,asciiName,attractionLights,availability,boosterTypes,borderColor,cardParts,colorIdentity,colorIndicator,...,subsets,subtypes,supertypes,text,toughness,type,types,uuid,variations,watermark
0,Pete Venters,d54c4a1a-c0c5-4834-84db-125d341f3ad8,NaN,NaN,"mtgo, paper",default,black,NaN,W,NaN,...,NaN,"Human, Cleric",NaN,First strike (This creature deals combat damag...,4,Creature — Human Cleric,Creature,5f8287b1-5bb6-5f4c-ad17-316a40d5bb0c,b7c19924-b4bf-56fc-aa73-f586e940bd42,NaN
1,Pete Venters,d54c4a1a-c0c5-4834-84db-125d341f3ad8,NaN,NaN,"mtgo, paper",default,black,NaN,W,NaN,...,NaN,"Human, Cleric",NaN,First strike (This creature deals combat damag...,4,Creature — Human Cleric,Creature,b7c19924-b4bf-56fc-aa73-f586e940bd42,5f8287b1-5bb6-5f4c-ad17-316a40d5bb0c,NaN
2,Volkan Baǵa,93bec3c0-0260-4d31-8064-5d01efb4153f,NaN,NaN,"mtgo, paper",default,black,NaN,W,NaN,...,NaN,Angel,NaN,"Flying\nWhen this creature enters, you gain 3 ...",3,Creature — Angel,Creature,57aaebc1-850c-503d-9f6e-bb8d00d8bf7c,8fd4e2eb-3eb4-50ea-856b-ef638fa47f8a,NaN
3,Volkan Baǵa,93bec3c0-0260-4d31-8064-5d01efb4153f,NaN,NaN,"mtgo, paper",default,black,NaN,W,NaN,...,NaN,Angel,NaN,"Flying\nWhen this creature enters, you gain 3 ...",3,Creature — Angel,Creature,8fd4e2eb-3eb4-50ea-856b-ef638fa47f8a,57aaebc1-850c-503d-9f6e-bb8d00d8bf7c,NaN
4,Mark Zug,48e2b98c-5467-4671-bd42-4c3746115117,NaN,NaN,"mtgo, paper",default,black,NaN,W,NaN,...,NaN,NaN,NaN,Target creature gets +3/+3 and gains flying un...,NaN,Sorcery,Sorcery,55bd38ca-dc73-5c06-8f80-a6ddd2f44382,c5655330-5131-5f40-9d3e-0549d88c6e9e,NaN


In [60]:
cards_df.columns

Index(['artist', 'artistIds', 'asciiName', 'attractionLights', 'availability',
       'boosterTypes', 'borderColor', 'cardParts', 'colorIdentity',
       'colorIndicator', 'colors', 'defense', 'duelDeck', 'edhrecRank',
       'edhrecSaltiness', 'faceConvertedManaCost', 'faceFlavorName',
       'faceManaValue', 'faceName', 'facePrintedName', 'finishes',
       'flavorName', 'flavorText', 'frameEffects', 'frameVersion', 'hand',
       'hasAlternativeDeckLimit', 'hasContentWarning', 'hasFoil', 'hasNonFoil',
       'isAlternative', 'isFullArt', 'isFunny', 'isGameChanger',
       'isOnlineOnly', 'isOversized', 'isPromo', 'isRebalanced', 'isReprint',
       'isReserved', 'isStarter', 'isStorySpotlight', 'isTextless',
       'isTimeshifted', 'keywords', 'language', 'layout', 'leadershipSkills',
       'life', 'loyalty', 'manaCost', 'manaValue', 'name', 'number',
       'originalPrintings', 'originalReleaseDate', 'originalText',
       'otherFaceIds', 'power', 'printedName', 'printedText', 'pr

In [61]:
# some of these columns only have True values, with 
# values that would otherwise be false being NaN.

# 'hasAlternativeDeckLimit', 'hasContentWarning' - too little count
# 'isFullArt', 'isFunny', 'isOversized' - unnecessary
# 'isStarter' - unnecessary AND deprecated
# 'isRebalanced', 'isStorySpotlight', 'isTextless', 'isTimeshifted' - not what we're looking for?

cards_df[
    [
        'isAlternative', 
        'isGameChanger', 
        'isOnlineOnly', 
        'isPromo', 
        'isReprint', 
        'isReserved'
        ]
        ] = cards_df[
            [
                'isAlternative', 
                'isGameChanger', 
                'isOnlineOnly', 
                'isPromo', 
                'isReprint', 
                'isReserved'
            ]
            ].fillna(False)

cards_df = cards_df[cards_df['isOnlineOnly'] == False]

In [62]:
# filter to paper and mtgo because price_df only lists those two
cards_df = cards_df[
    (
        cards_df['availability'].str.contains('paper')
    ) | (
            cards_df['availability'].str.contains('mtgo')
        )
    ]
cards_df['availability'].value_counts()

availability
mtgo, paper           40854
paper                 38162
arena, mtgo, paper    17978
arena, paper           1158
Name: count, dtype: int64

In [63]:
cards_df_eng = cards_df[cards_df['language'] == 'English']

# don't need artists?
# do we need borderColor? flavorText?
# otherFaceIds - for split

cards_df_feats = cards_df_eng[[
                            'uuid', 
                            'name', 
                            'availability',
                            'colorIdentity', 
                            'defense', 
                            'edhrecRank', 
                            'edhrecSaltiness', 
                            'finishes', 
                            'isAlternative', 
                            'isGameChanger', 
                            'isPromo', 
                            'isReprint', 
                            'isReserved', 
                            'keywords', 
                            'layout', 
                            'loyalty', 
                            'manaCost',
                            'manaValue', 
                            'number', 
                            'otherFaceIds', 
                            'power', 
                            'rarity', 
                            'setCode', 
                            'subtypes', 
                            'supertypes', 
                            'text', 
                            'toughness', 
                            'types', 
                            'variations'
                            ]]

cards_df_feats = cards_df_feats.rename(columns={'name': 'cardName', 'number': 'cardNumber'})

cards_df_feats.head()

,uuid,cardName,availability,colorIdentity,defense,edhrecRank,edhrecSaltiness,finishes,isAlternative,isGameChanger,...,otherFaceIds,power,rarity,setCode,subtypes,supertypes,text,toughness,types,variations
0,5f8287b1-5bb6-5f4c-ad17-316a40d5bb0c,Ancestor's Chosen,"mtgo, paper",W,NaN,23684.0,0.27,nonfoil,False,False,...,NaN,4,uncommon,10E,"Human, Cleric",NaN,First strike (This creature deals combat damag...,4,Creature,b7c19924-b4bf-56fc-aa73-f586e940bd42
1,b7c19924-b4bf-56fc-aa73-f586e940bd42,Ancestor's Chosen,"mtgo, paper",W,NaN,23684.0,0.27,foil,False,False,...,NaN,4,uncommon,10E,"Human, Cleric",NaN,First strike (This creature deals combat damag...,4,Creature,5f8287b1-5bb6-5f4c-ad17-316a40d5bb0c
2,57aaebc1-850c-503d-9f6e-bb8d00d8bf7c,Angel of Mercy,"mtgo, paper",W,NaN,18663.0,NaN,nonfoil,False,False,...,NaN,3,uncommon,10E,Angel,NaN,"Flying\nWhen this creature enters, you gain 3 ...",3,Creature,8fd4e2eb-3eb4-50ea-856b-ef638fa47f8a
3,8fd4e2eb-3eb4-50ea-856b-ef638fa47f8a,Angel of Mercy,"mtgo, paper",W,NaN,18663.0,NaN,foil,False,False,...,NaN,3,uncommon,10E,Angel,NaN,"Flying\nWhen this creature enters, you gain 3 ...",3,Creature,57aaebc1-850c-503d-9f6e-bb8d00d8bf7c
4,55bd38ca-dc73-5c06-8f80-a6ddd2f44382,Angelic Blessing,"mtgo, paper",W,NaN,25093.0,0.19,nonfoil,False,False,...,NaN,NaN,common,10E,NaN,NaN,Target creature gets +3/+3 and gains flying un...,NaN,Sorcery,c5655330-5131-5f40-9d3e-0549d88c6e9e


In [64]:
prices_df = pd.read_csv('data/mtgjson.com/cardPrices.csv')

prices_df = prices_df[prices_df['currency'] == 'USD']

print(prices_df['priceProvider'].value_counts())

prices_df.head()

priceProvider
cardkingdom    208497
manapool       142758
tcgplayer      142424
cardhoarder     79777
cardsphere      76114
Name: count, dtype: int64


,cardFinish,currency,date,gameAvailability,price,priceProvider,providerListing,uuid
0,normal,USD,2026-01-03,mtgo,0.18,cardhoarder,retail,f182e364-0439-5594-a6e6-75f7889ccf45
1,normal,USD,2026-01-03,mtgo,0.33,cardhoarder,retail,330deaa3-dd7a-52a8-bfbc-b323cd16a409
2,normal,USD,2026-01-03,mtgo,0.02,cardhoarder,retail,79e36956-b91f-580f-8309-7d9585a67560
3,normal,USD,2026-01-03,mtgo,0.33,cardhoarder,retail,6afb2b4c-530a-57d5-8e7f-871239f6fa05
4,normal,USD,2026-01-03,mtgo,0.02,cardhoarder,retail,b1fc2762-92aa-5a14-8509-a59cb611e376


In [65]:
average_prices = prices_df.groupby(['uuid'])['price'].mean().reset_index()
average_prices.head()

,uuid,price
0,00010d56-fe38-5e35-8aed-518019aa36a5,7.910000
1,0001e0d0-2dcd-5640-aadc-a84765cf5fc9,4.715000
2,0003caab-9ff5-5d1a-bc06-976dd0457f19,0.539000
3,0003d249-25d9-5223-af1e-1130f09622a7,0.234444
4,0004822c-c181-5564-808d-a6cc48359a1a,1.063333


In [66]:
legal_df = pd.read_csv('data/mtgjson.com/cardLegalities.csv')

legal_df.head()

C:\Users\sophc\AppData\Local\Temp\ipykernel_48256\2955574431.py:1: DtypeWarning: Columns (0: oldschool) have mixed types. Specify dtype option on import or set low_memory=False.
  legal_df = pd.read_csv('data/mtgjson.com/cardLegalities.csv')


,alchemy,brawl,commander,duel,future,gladiator,historic,legacy,modern,oathbreaker,...,paupercommander,penny,pioneer,predh,premodern,standard,standardbrawl,timeless,uuid,vintage
0,NaN,NaN,Legal,Legal,NaN,NaN,NaN,Legal,Legal,Legal,...,NaN,Legal,NaN,Legal,Legal,NaN,NaN,NaN,5f8287b1-5bb6-5f4c-ad17-316a40d5bb0c,Legal
1,NaN,NaN,Legal,Legal,NaN,NaN,NaN,Legal,Legal,Legal,...,NaN,Legal,NaN,Legal,Legal,NaN,NaN,NaN,b7c19924-b4bf-56fc-aa73-f586e940bd42,Legal
2,NaN,Legal,Legal,Legal,NaN,Legal,Legal,Legal,Legal,Legal,...,Legal,Legal,NaN,Legal,Legal,NaN,NaN,Legal,57aaebc1-850c-503d-9f6e-bb8d00d8bf7c,Legal
3,NaN,Legal,Legal,Legal,NaN,Legal,Legal,Legal,Legal,Legal,...,Legal,Legal,NaN,Legal,Legal,NaN,NaN,Legal,8fd4e2eb-3eb4-50ea-856b-ef638fa47f8a,Legal
4,NaN,NaN,Legal,Legal,NaN,NaN,NaN,Legal,Legal,Legal,...,Legal,Legal,NaN,Legal,Legal,NaN,NaN,NaN,55bd38ca-dc73-5c06-8f80-a6ddd2f44382,Legal


In [67]:
# keep it to formats that have many legal cards
# but also seem semi-popular, needs review

print(legal_df['commander'].value_counts())
print(legal_df['legacy'].value_counts())
print(legal_df['standard'].value_counts())

legal_df_most_cards = legal_df[['commander', 'legacy', 'standard', 'vintage', 'uuid']]

commander
Legal     101057
Banned       350
Name: count, dtype: int64
legacy
Legal     100375
Banned       897
Name: count, dtype: int64
standard
Legal     16144
Banned       28
Name: count, dtype: int64


In [68]:
sets_df = pd.read_csv('data/mtgjson.com/sets.csv')

# needed for release date

sets_df.head()

,baseSetSize,block,cardsphereSetId,code,isFoilOnly,isForeignOnly,isNonFoilOnly,isOnlineOnly,isPartialPreview,keyruneCode,...,mcmIdExtras,mcmName,mtgoCode,name,parentCode,releaseDate,tcgplayerGroupId,tokenSetCode,totalSetSize,type
0,383,Core Set,755.0,10E,False,NaN,NaN,False,NaN,10E,...,NaN,Tenth Edition,10E,Tenth Edition,NaN,2007-07-13,1.0,T10E,510,core
1,302,Core Set,938.0,2ED,False,NaN,True,False,NaN,2ED,...,NaN,NaN,NaN,Unlimited Edition,NaN,1993-12-01,115.0,NaN,302,core
2,331,NaN,1462.0,2X2,False,NaN,NaN,False,NaN,2X2,...,5071.0,Double Masters 2022,NaN,Double Masters 2022,NaN,2022-07-08,3070.0,T2X2,579,masters
3,332,NaN,1251.0,2XM,False,NaN,NaN,False,NaN,2XM,...,3209.0,Double Masters,2XM,Double Masters,NaN,2020-08-07,2655.0,T2XM,384,masters
4,594,NaN,NaN,30A,False,NaN,True,False,NaN,30A,...,NaN,30th Anniversary Edition,NaN,30th Anniversary Edition,NaN,2022-11-28,3178.0,T30A,594,memorabilia


In [69]:
sets_df_real_eng = sets_df[(sets_df['isPartialPreview'] != True) & 
                        (sets_df['isForeignOnly'] != True) & 
                        (sets_df['isOnlineOnly'] != True)]

# "type": "expType"
# don't need type?

sets_df_real_eng = sets_df_real_eng[['code', 'name', 'releaseDate']]
sets_df_real_eng = sets_df_real_eng.rename(columns={"code": "setCode", "name": "setName"})
sets_df_real_eng.head()

,setCode,setName,releaseDate
0,10E,Tenth Edition,2007-07-13
1,2ED,Unlimited Edition,1993-12-01
2,2X2,Double Masters 2022,2022-07-08
3,2XM,Double Masters,2020-08-07
4,30A,30th Anniversary Edition,2022-11-28


In [70]:
card_ids_df = pd.read_csv("data/mtgjson.com/cardIdentifiers.csv")

# 'cardKingdomId', 'cardsphereId' are priceProviders in price_df
# but price_df will be averaged, so not needed?

# keep TCGPlayer and Multiverse (official Gatherer tool)
# in case to get their data

card_ids_df = card_ids_df[['uuid', 'scryfallId', 'tcgplayerProductId', 'multiverseId']]

card_ids_df.head()

C:\Users\sophc\AppData\Local\Temp\ipykernel_48256\98840952.py:1: DtypeWarning: Columns (0: mtgjsonFoilVersionId, 1: mtgjsonNonFoilVersionId) have mixed types. Specify dtype option on import or set low_memory=False.
  card_ids_df = pd.read_csv("data/mtgjson.com/cardIdentifiers.csv")


,uuid,scryfallId,tcgplayerProductId,multiverseId
0,5f8287b1-5bb6-5f4c-ad17-316a40d5bb0c,7a5cd03c-4227-4551-aa4b-7d119f0468b5,15032.0,130550.0
1,b7c19924-b4bf-56fc-aa73-f586e940bd42,82072a1d-c1ab-4b4f-875f-d0591447e0a4,15032.0,NaN
2,57aaebc1-850c-503d-9f6e-bb8d00d8bf7c,8f7980d4-da43-4d6d-ad16-14b8a34ae91d,15033.0,129465.0
3,8fd4e2eb-3eb4-50ea-856b-ef638fa47f8a,b0157252-6949-4f03-a15c-c168512123a8,15033.0,NaN
4,55bd38ca-dc73-5c06-8f80-a6ddd2f44382,a285aa3f-bcfb-4fc3-8441-85a56c72a3e4,15035.0,129711.0


In [71]:
print(len(cards_df_feats), len(average_prices))

card_prices_df = cards_df_feats.merge(average_prices, how='right', on='uuid')
print(len(card_prices_df))

95685 104075
104075


In [72]:
card_prices_legal_df = card_prices_df.merge(legal_df_most_cards, on='uuid')
print(len(card_prices_legal_df))

99736


In [73]:
card_prices_sets_df = card_prices_legal_df.merge(sets_df_real_eng, how='left', on='setCode')
print(len(card_prices_sets_df))

99736


In [74]:
card_prices_sets_ids_df = card_prices_sets_df.merge(card_ids_df, on='uuid')
print(len(card_prices_sets_ids_df))

99736


In [75]:
card_prices_sets_legal_df = card_prices_sets_ids_df[
    (card_prices_sets_ids_df['commander'] == 'Legal') |
    (card_prices_sets_ids_df['legacy'] == 'Legal') |
    (card_prices_sets_ids_df['standard'] == 'Legal')
    ]

# I don't think having unreleased cards is good
card_prices_sets_legal_df = card_prices_sets_legal_df[
    card_prices_sets_legal_df['releaseDate'].notnull()
    ]

print(len(card_prices_sets_legal_df))
card_prices_sets_legal_df.head()


89570


,uuid,cardName,availability,colorIdentity,defense,edhrecRank,edhrecSaltiness,finishes,isAlternative,isGameChanger,...,price,commander,legacy,standard,vintage,setName,releaseDate,scryfallId,tcgplayerProductId,multiverseId
0,00010d56-fe38-5e35-8aed-518019aa36a5,Sphinx of the Final Word,paper,U,NaN,10186.0,0.14,foil,False,False,...,7.910000,Legal,Legal,Legal,Legal,Oath of the Gatewatch Promos,2016-01-22,f6555d1f-d4cf-41f7-99d3-88fd53e75457,111268.0,NaN
1,0001e0d0-2dcd-5640-aadc-a84765cf5fc9,Goblin King,paper,R,NaN,3454.0,0.34,nonfoil,False,False,...,4.715000,Legal,Legal,NaN,Legal,Revised Edition,1994-04-11,e3094187-d666-414b-a1fd-ae0ef55c3fcb,1435.0,1296.0
2,0003caab-9ff5-5d1a-bc06-976dd0457f19,Caravan Vigil,"mtgo, paper",G,NaN,12489.0,0.08,"nonfoil, foil",False,False,...,0.539000,Legal,Legal,NaN,Legal,Innistrad,2011-09-30,9a8dfb98-a975-41bf-8aac-c0001c9ddaa7,56309.0,234444.0
3,0003d249-25d9-5223-af1e-1130f09622a7,Deadshot Minotaur,"mtgo, paper","G, R",NaN,25150.0,0.20,"nonfoil, foil",False,False,...,0.234444,Legal,Legal,NaN,Legal,Alara Reborn,2009-04-30,aacb131b-74c9-4e6c-9466-27710bc9441f,31714.0,179543.0
4,0004822c-c181-5564-808d-a6cc48359a1a,Clone Legion,"arena, mtgo, paper",U,NaN,3488.0,0.48,"nonfoil, foil",False,False,...,1.063333,Legal,Legal,NaN,Legal,Avatar: The Last Airbender Eternal,2025-11-21,b7ea0b25-19e3-4d75-9daa-9110dce7e6fd,661987.0,NaN


In [76]:
card_prices_sets_legal_df.columns

Index(['uuid', 'cardName', 'availability', 'colorIdentity', 'defense',
       'edhrecRank', 'edhrecSaltiness', 'finishes', 'isAlternative',
       'isGameChanger', 'isPromo', 'isReprint', 'isReserved', 'keywords',
       'layout', 'loyalty', 'manaCost', 'manaValue', 'cardNumber',
       'otherFaceIds', 'power', 'rarity', 'setCode', 'subtypes', 'supertypes',
       'text', 'toughness', 'types', 'variations', 'price', 'commander',
       'legacy', 'standard', 'vintage', 'setName', 'releaseDate', 'scryfallId',
       'tcgplayerProductId', 'multiverseId'],
      dtype='str')

In [77]:
print(card_prices_sets_legal_df['isReprint'].value_counts())
print('---')
print(card_prices_sets_legal_df['availability'].value_counts())
print('---')
print(card_prices_sets_legal_df['layout'].value_counts())

isReprint
True     50189
False    39381
Name: count, dtype: int64
---
availability
mtgo, paper           40629
paper                 30053
arena, mtgo, paper    17731
arena, paper           1157
Name: count, dtype: int64
---
layout
normal             85057
transform           1926
adventure            670
modal_dfc            478
split                418
saga                 343
reversible_card      150
aftermath            140
mutate               101
flip                  62
class                 61
leveler               60
meld                  42
prototype             38
case                  24
Name: count, dtype: int64


In [78]:
print(card_prices_sets_legal_df['types'].value_counts())

types
Creature                  39909
Land                       9964
Instant                    9578
Sorcery                    9551
Enchantment                8318
Artifact                   7031
Artifact, Creature         2810
Planeswalker               1295
Enchantment, Creature       708
Artifact, Land              111
Kindred, Instant             53
Battle                       53
Kindred, Enchantment         51
Stickers                     48
Kindred, Sorcery             38
Enchantment, Artifact        20
Enchantment, Land            12
Kindred, Artifact             9
Land, Creature                6
Artifact, Planeswalker        5
Name: count, dtype: int64


In [79]:
print(card_prices_sets_legal_df['rarity'].value_counts())
print('---')
print(card_prices_sets_legal_df['subtypes'].value_counts())
print('---')
print(card_prices_sets_legal_df['supertypes'].value_counts())

rarity
rare        33851
common      25073
uncommon    21791
mythic       8497
special       358
Name: count, dtype: int64
---
subtypes
Aura                    2666
Equipment               1490
Human, Wizard           1441
Human, Soldier          1223
Elemental                979
                        ... 
Homarid, Scout             1
Ooze, Brushwagg            1
Jellyfish, Artificer       1
Djinn, Artificer           1
Mole, Scout                1
Name: count, Length: 2523, dtype: int64
---
supertypes
Legendary          12676
Basic               3270
Snow                 175
Basic, Snow           55
World                 37
Legendary, Snow       30
Name: count, dtype: int64


In [80]:
# some cards just don't have [x] property, hence they have nan values
# certain features only apply to specific card types

# colorIdentity - 9151 nan
# defense - 89517 nan
# edhrecRank - 3457 nan
# edhrecSaltiness - 10240 nan
# keywords - 50535 nan
# loyalty - 88281 nan
# manaCost - 11200 nan
# otherFaceIds - 85690 nan
# power - 45589 nan
# subtypes - 33921 nan
# supertypes - 73327 nan
# text - 904 nan
# toughness - 45589 nan
# variations - 58935 nan
# legacy - 135 nan
# standard - 74896 nan
# vintage - 135 nan
# tcgplayerProductId - 1024 nan
# multiverseId - 23957 nan

# card_prices_sets_legal_df.isna().sum()

In [81]:
card_prices_sets_legal_df[['types', 'supertypes', 'subtypes']]

,types,supertypes,subtypes
0,Creature,NaN,Sphinx
1,Creature,NaN,Goblin
2,Sorcery,NaN,NaN
3,Creature,NaN,Minotaur
4,Sorcery,NaN,NaN
...,...,...,...
99730,Artifact,NaN,NaN
99731,Creature,NaN,"Human, Monk"
99732,Creature,NaN,"Elf, Knight"
99733,Land,Basic,Island


In [82]:
# humans are common, what nonhuman subtypes have the most entries
card_prices_sets_legal_df[
    (~card_prices_sets_legal_df['subtypes'].str.contains('Human')) & 
    (card_prices_sets_legal_df['types'].str.contains('Creature'))
    ]['subtypes'].value_counts().head(10)

subtypes
Elemental     971
Dragon        927
Spirit        865
Angel         748
Beast         641
Zombie        554
Construct     513
Vampire       411
Elf, Druid    410
Demon         406
Name: count, dtype: int64

In [83]:
beast_count = card_prices_sets_legal_df[
    card_prices_sets_legal_df['subtypes'].str.contains('Beast')
    ]['subtypes'].value_counts()

print(beast_count.sum())
print(beast_count.head(10))

1144
subtypes
Beast               641
Cat, Beast           59
Phyrexian, Beast     51
Boar, Beast          24
Elemental, Beast     23
Dinosaur, Beast      22
Beast, Noble         21
Beast, Horror        16
Zombie, Beast        15
Fungus, Beast        13
Name: count, dtype: int64


In [84]:
construct_count = card_prices_sets_legal_df[card_prices_sets_legal_df['subtypes'].str.contains('Construct')
                                      ]['subtypes'].value_counts()
print(construct_count.sum())
print(construct_count.head(10))

588
subtypes
Construct               513
Phyrexian, Construct     36
Myr, Construct           15
Bird, Construct           6
Wolf, Construct           4
Horror, Construct         4
Saga, Construct           2
Demon, Construct          2
Golem, Construct          1
Goblin, Construct         1
Name: count, dtype: int64


In [85]:
demon_count = card_prices_sets_legal_df[card_prices_sets_legal_df['subtypes'].str.contains('Demon')
                                      ]['subtypes'].value_counts()
print(demon_count.sum())
print(demon_count.head(10))

656
subtypes
Demon               408
Demon, Spirit        28
Elder, Demon         16
Demon, Berserker     13
Vampire, Demon       12
Phyrexian, Demon     11
Demon, Dragon        11
Avatar, Demon        10
Demon, Cleric        10
Ogre, Demon           9
Name: count, dtype: int64


In [86]:
elf_count = card_prices_sets_legal_df[
    card_prices_sets_legal_df['subtypes'].str.contains('Elf')
    ]['subtypes'].value_counts()

print(elf_count.sum())
print(elf_count.head(10))

1913
subtypes
Elf, Druid      410
Elf, Warrior    251
Elf, Shaman     225
Elf             158
Elf, Scout      133
Elf, Archer      81
Elf, Noble       72
Elf, Wizard      57
Elf, Knight      30
Elf, Rogue       29
Name: count, dtype: int64


In [87]:
card_prices_sets_legal_df[card_prices_sets_legal_df['subtypes'].str.contains('Human')
                          ]['subtypes'].value_counts().head(10)

subtypes
Human, Wizard       1441
Human, Soldier      1223
Human, Knight        751
Human, Cleric        720
Human, Warrior       696
Human, Rogue         396
Human, Shaman        375
Human, Artificer     351
Human, Druid         295
Human                234
Name: count, dtype: int64

In [88]:
artificer_count = card_prices_sets_legal_df[
    card_prices_sets_legal_df[
        'subtypes'
        ].str.contains(
        'Artificer'
        )
        ]['subtypes'].value_counts()

print(artificer_count.sum())
print(artificer_count.head(10))

722
subtypes
Human, Artificer               351
Goblin, Artificer               61
Vedalken, Artificer             37
Dwarf, Artificer                37
Elf, Artificer                  26
Phyrexian, Human, Artificer     25
Human, Rogue, Artificer         18
Phyrexian, Artificer            17
Faerie, Artificer               13
Kor, Artificer                  12
Name: count, dtype: int64


In [89]:
card_prices_sets_legal_df['keywords'].value_counts().head(10)

keywords
Flying       4385
Enchant      2393
Equip        1194
Scry          978
Trample       943
Mill          828
Cycling       735
Haste         613
Vigilance     599
Flashback     598
Name: count, dtype: int64

In [90]:
# boolean/dummy vars for common subtypes
final_df = card_prices_sets_legal_df.copy()

# heritage
final_df['isHuman'] = final_df['subtypes'].str.contains('Human')
final_df['isElemental'] = final_df['subtypes'].str.contains('Elemental')
final_df['isDragon'] = final_df['subtypes'].str.contains('Dragon')
final_df['isSpirit'] = final_df['subtypes'].str.contains('Spirit')
final_df['isAngel'] = final_df['subtypes'].str.contains('Angel')
final_df['isElf'] = final_df['subtypes'].str.contains('Elf')
final_df['isVampire'] = final_df['subtypes'].str.contains('Vampire')
final_df['isZombie'] = final_df['subtypes'].str.contains('Zombie')
final_df['isBeast'] = final_df['subtypes'].str.contains('Beast')

# class
final_df['isWizard'] = final_df['subtypes'].str.contains('Wizard')
final_df['isSoldier'] = final_df['subtypes'].str.contains('Soldier')
final_df['isKnight'] = final_df['subtypes'].str.contains('Knight')
final_df['isCleric'] = final_df['subtypes'].str.contains('Cleric')
final_df['isWarrior'] = final_df['subtypes'].str.contains('Warrior')
final_df['isRogue'] = final_df['subtypes'].str.contains('Rogue')
final_df['isShaman'] = final_df['subtypes'].str.contains('Shaman')
final_df['isDruid'] = final_df['subtypes'].str.contains('Druid')

# other
final_df['isCreature'] = final_df['types'].str.contains('Creature')
final_df['isPlaneswalker'] = final_df['types'].str.contains('Planeswalker')
final_df['isMTGO'] = final_df['availability'].str.contains('mtgo')

# keywords
final_df['isFlying'] = final_df['keywords'].str.contains('Flying')

In [91]:
final_df.columns

Index(['uuid', 'cardName', 'availability', 'colorIdentity', 'defense',
       'edhrecRank', 'edhrecSaltiness', 'finishes', 'isAlternative',
       'isGameChanger', 'isPromo', 'isReprint', 'isReserved', 'keywords',
       'layout', 'loyalty', 'manaCost', 'manaValue', 'cardNumber',
       'otherFaceIds', 'power', 'rarity', 'setCode', 'subtypes', 'supertypes',
       'text', 'toughness', 'types', 'variations', 'price', 'commander',
       'legacy', 'standard', 'vintage', 'setName', 'releaseDate', 'scryfallId',
       'tcgplayerProductId', 'multiverseId', 'isHuman', 'isElemental',
       'isDragon', 'isSpirit', 'isAngel', 'isElf', 'isVampire', 'isZombie',
       'isBeast', 'isWizard', 'isSoldier', 'isKnight', 'isCleric', 'isWarrior',
       'isRogue', 'isShaman', 'isDruid', 'isCreature', 'isPlaneswalker',
       'isMTGO', 'isFlying'],
      dtype='str')

In [92]:
final_df['power'] = pd.to_numeric(final_df['power'], downcast='integer', errors='coerce')
final_df['toughness'] = pd.to_numeric(final_df['toughness'], downcast='integer', errors='coerce')
final_df['loyalty'] = pd.to_numeric(final_df['loyalty'], downcast='integer', errors='coerce')

In [93]:
final_df.to_csv("data/finalCards.csv", index=False)

In [94]:
final_df.columns

Index(['uuid', 'cardName', 'availability', 'colorIdentity', 'defense',
       'edhrecRank', 'edhrecSaltiness', 'finishes', 'isAlternative',
       'isGameChanger', 'isPromo', 'isReprint', 'isReserved', 'keywords',
       'layout', 'loyalty', 'manaCost', 'manaValue', 'cardNumber',
       'otherFaceIds', 'power', 'rarity', 'setCode', 'subtypes', 'supertypes',
       'text', 'toughness', 'types', 'variations', 'price', 'commander',
       'legacy', 'standard', 'vintage', 'setName', 'releaseDate', 'scryfallId',
       'tcgplayerProductId', 'multiverseId', 'isHuman', 'isElemental',
       'isDragon', 'isSpirit', 'isAngel', 'isElf', 'isVampire', 'isZombie',
       'isBeast', 'isWizard', 'isSoldier', 'isKnight', 'isCleric', 'isWarrior',
       'isRogue', 'isShaman', 'isDruid', 'isCreature', 'isPlaneswalker',
       'isMTGO', 'isFlying'],
      dtype='str')